[⬅ Back to the examples](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/examples/)

# First-order process with PID tuning

A first-order plant $G(s) = \dfrac{K}{\tau s + 1}$ is controlled by a PID
controller (parallel form, derivative on measurement, anti-windup). Tune the
gains with the knobs while the simulation runs, change the setpoint, and watch
the response on the strip chart.

*Front panel*: gains **Kp**, **Ti**, **Td**, setpoint slide, process gain and
time constant, controller output gauge, RUN switch, STEP button (latched) and
an emergency stop that forces the output to 0 %.

> **The code is hidden.** To see a cell's code, click the `⋯` bar above its result; to see all of it, choose **View ▸ Expand All Code**.

**References:** the emergency stop only borrows the look of IEC 60204-1 /
ISO 13850 devices (see the safety notice). These standards inspired the
widgets; the library does not claim conformity with them ([Standards and
references](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/standards/)).

In [ ]:
import os
import threading
import time

import numpy as np

import anywidget_instruments_industrial as ai

# How long the live simulation runs (seconds). The documentation build and the
# test-suite set AWI_EXAMPLE_SECONDS to run the examples briefly.
RUN_SECONDS = float(os.environ.get("AWI_EXAMPLE_SECONDS", "600"))

In [ ]:
ai.theme_switch()  # light / system / dark theme (STYLE-008)

In [ ]:
kp = ai.Knob(2.0, min=0, max=10, step=0.1, label="Kp", size=(110, 110))
ti = ai.Knob(5.0, min=0.1, max=30, step=0.1, unit="s", label="Ti", size=(110, 110))
td = ai.Knob(0.0, min=0, max=5, step=0.05, unit="s", label="Td", size=(110, 110))
sp = ai.FillSlide(50, min=0, max=100, step=1, unit="%", label="Setpoint", size=(240, 70))
gain = ai.Knob(1.5, min=0.1, max=5, step=0.1, label="Process K", size=(110, 110))
tau = ai.Knob(8.0, min=0.5, max=30, step=0.5, unit="s", label="Process τ", size=(110, 110))
out = ai.Gauge(
    0, min=0, max=100, unit="%", label="Output", hi=95, show_limits=True, size=(150, 150)
)
pv_display = ai.SevenSegment(0, digits=5, decimals=1, label="PV", size=(160, 64))
run = ai.ToggleSwitch(True, label="RUN")
step_btn = ai.PushButton(text="STEP +10", mechanical_action="latch_when_released", label="Step")
estop = ai.EmergencyStop(label="E-STOP")
chart = ai.WaveformChart(
    n_traces=3,
    history=1200,
    dt=0.05,
    x_unit="s",
    unit="%",
    y_min=-5,
    y_max=110,
    size=(640, 260),
    label="Response",
    traces=[{"name": "SP"}, {"name": "PV"}, {"name": "OUT", "width": 1}],
)

ai.Panel(
    [
        ai.Panel([kp, ti, td, gain, tau], columns=5),
        ai.Panel([sp, out, pv_display, run, step_btn, estop], columns=6),
        chart,
    ],
    columns=1,
)

In [ ]:
class PID:
    """Parallel PID with derivative on measurement and clamping anti-windup."""

    def __init__(self):
        self.integral = 0.0
        self.last_pv = None

    def update(self, sp, pv, dt, kp, ti, td):
        error = sp - pv
        d_pv = 0.0 if self.last_pv is None else (pv - self.last_pv) / dt
        self.last_pv = pv
        u = kp * (error + self.integral / ti - td * d_pv)
        if 0.0 < u < 100.0:  # integrate only when not saturated (anti-windup)
            self.integral += error * dt
        return min(max(u, 0.0), 100.0)


def plant_step(y, u, dt, k, tau):
    """Exact discretization of K/(tau s + 1) for a constant input over dt."""
    a = np.exp(-dt / tau)
    return a * y + k * (1 - a) * u


def simulate(duration=RUN_SECONDS, dt=0.05):
    pid, y = PID(), 0.0
    t_end = time.time() + duration
    while time.time() < t_end:
        if step_btn.read_latched():
            sp.value = min(sp.max, sp.value + 10)
        if estop.value:
            u = 0.0
        elif run.value:
            u = pid.update(sp.value, y, dt, kp.value, ti.value, td.value)
        else:
            u = out.value  # hold
        y = plant_step(y, u, dt, gain.value, tau.value)
        with ai.batch():
            out.value = u
            pv_display.value = y
            chart.append([sp.value, y, u])
        time.sleep(dt)


threading.Thread(target=simulate, daemon=True).start()

**Things to try**

* Increase **Kp** until the response overshoots, then add integral action.
* Set **Td** to damp the overshoot of an aggressive tuning.
* Pause the chart (`chart.paused = True`) and use the zoom tool and the
  cursors to measure rise time and overshoot.
* Press **E-STOP**: the output drops to 0 %; `estop.reset()` releases it.

In [ ]:
# Measure the step response with cursors (chart.cursor_values holds the values)
chart.add_cursor(10.0, name="t1")